For fixed diffusion co-efficients we control the upward velocities and directionality to simulate the three cases: still, breeze and stormy.

For the behavioural component, the orientation (phi) which is originally controlled by the rotational diffusion co-efficient, is now indepent of it and manually controlled.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML

Defining parameters

In [ ]:
n_planktons = 50
arena_length = 100
n_frames = 100
Dr = 3
Dt = 10
delta_t = 0.1
velocity = 3

In [ ]:
Kb = 1.38e-23
T = 300
eta = 1e-3
R = 1000e-6

real_Dr = Kb*T/(8 * np.pi * eta * R**3)
real_Dt = Kb*T/(6 * np.pi * eta * R)

print("Dr", real_Dr)
print("Dt", real_Dt)

In [ ]:
Dt_turbulence = 20
Dr_turbulence = 20

For behavioral studies, we first define a target angle indicating the location of the light source. In this case, it is 90 degress or pi/2 radians.

The possible orientations for each case, still, breeze and stormy around this target angle as follows:

- +- 5 degrees for still
- +- 30 degrees for breeze
- +- 60 degrees for stormy

In [ ]:
target_angle = np.pi / 2
response_angle = 180 * (np.pi / 180)

Initialising starting values

In [ ]:
x = np.random.rand(n_planktons) * 2 * arena_length - arena_length
y = np.random.rand(n_planktons) * 2 * arena_length - arena_length
phi = np.random.rand(n_planktons) * 2 * np.pi

# History of positions (used for trails)
trail_length = 20
history_x = np.full((trail_length, n_planktons), np.nan)
history_y = np.full((trail_length, n_planktons), np.nan)

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
scat = ax.scatter(x, y, c='white', s=20, alpha=0.5)
trails = [ax.plot([], [], '-', linewidth=1, alpha=0.5)[0] for _ in range(n_planktons)]
ax.set_xlim(-arena_length, arena_length)
ax.set_ylim(-arena_length, arena_length)
ax.set_facecolor('black')

In [ ]:
plt.hist(y, bins=20)
plt.xlim(-arena_length, arena_length)
plt.show()

In [ ]:
def update(frame):

    global x, y, phi,  history_x, history_y
    # time_steps = 500

    phi = phi + np.sqrt(2 * Dr * delta_t) * np.random.randn(n_planktons) + np.sqrt(2 * Dr_turbulence * delta_t) * np.random.randn(n_planktons) # normal SDE + turbulence

    x = x + velocity * np.cos(phi) * delta_t + np.sqrt(2 * Dt * delta_t) * np.random.randn(n_planktons) + np.sqrt(2 * Dt_turbulence * delta_t) * np.random.randn(n_planktons)
    y = y + velocity * np.sin(phi) * delta_t + np.sqrt(2 * Dt * delta_t) * np.random.randn(n_planktons) + np.sqrt(2 * Dt_turbulence * delta_t) * np.random.randn(n_planktons)

    # Reflect at walls
    y[y > arena_length] = 2 * arena_length - y[y > arena_length]
    y[y < -arena_length] = -2 * arena_length - y[y < -arena_length]
    x[x > arena_length] = 2 * arena_length - x[x > arena_length]
    x[x < -arena_length] = -2 * arena_length - x[x < -arena_length]

    scat.set_offsets(np.c_[x, y]) 

    # Update trails
    history_x = np.roll(history_x, -1, axis=0)
    history_y = np.roll(history_y, -1, axis=0)
    history_x[-1, :] = x
    history_y[-1, :] = y

    for i, trail in enumerate(trails):
        trail.set_data(history_x[:, i], history_y[:, i])   

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
scat = ax.scatter(x, y, c='white', s=20, alpha=0.5)
trails = [ax.plot([], [], '-', linewidth=1, alpha=0.5)[0] for _ in range(n_planktons)]
ax.set_xlim(-arena_length, arena_length)
ax.set_ylim(-arena_length, arena_length)
ax.set_facecolor('black')

In [ ]:
plt.hist(y, bins=20)
plt.xlim(-arena_length, arena_length)
plt.show()

In [ ]:
ani = FuncAnimation(fig, update, frames=100, interval=100)
HTML(ani.to_jshtml())

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Parameters
N = 50                     # Number of Fourier modes
grid_size = 20             # Grid size for the velocity field (20x20 points)
L = 1.0                    # Length scale of the simulation domain
c = 1.0                    # Constant for frequency relation
k_min = 2 * np.pi / L      # Minimum wavenumber (based on domain size)
k_max = 10 * k_min         # Maximum wavenumber
amplitude_scale = 0.1      # Scaling factor for mode amplitudes

# Generate random wavevectors, amplitudes, and phases
np.random.seed(0)  # Set seed for reproducibility
wavevectors = k_min + (k_max - k_min) * np.random.rand(N)
angles = 2 * np.pi * np.random.rand(N)
kx = wavevectors * np.cos(angles)  # x-component of wavevector
ky = wavevectors * np.sin(angles)  # y-component of wavevector
amplitudes = amplitude_scale * wavevectors**(-5/6)  # Kolmogorov scaling

# Grid setup for visualization
x = np.linspace(0, L, grid_size)
y = np.linspace(0, L, grid_size)
X, Y = np.meshgrid(x, y)

# Time-dependent velocity field calculation function
def calculate_velocity_field(X, Y, t):
    u = np.zeros(X.shape)  # Initialize u component of velocity
    v = np.zeros(Y.shape)  # Initialize v component of velocity
    for i in range(N):
        # Calculate omega for each mode
        omega = c * np.sqrt(kx[i]**2 + ky[i]**2)
        
        # Random phase for each mode
        phase = 2 * np.pi * np.random.rand()
        
        # Update velocity field with this mode
        u += amplitudes[i] * np.cos(kx[i] * X + ky[i] * Y - omega * t + phase)
        v += amplitudes[i] * np.sin(kx[i] * X + ky[i] * Y - omega * t + phase)
    
    return u, v

# Visualization at a specific time step
t = 0  # Set time to visualize
u, v = calculate_velocity_field(X, Y, t)

# Plotting the velocity field using quiver plot
plt.figure(figsize=(8, 8))
plt.quiver(X, Y, u, v, scale=1.5, color='blue')
plt.title(f'Isotropic Turbulent Velocity Field at t = {t}')
plt.xlabel('x')
plt.ylabel('y')
plt.axis('equal')
plt.grid()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Parameters
N = 50                     # Number of Fourier modes
grid_size = 100            # Grid size for the velocity field (20x20 points)
L = 1.0                    # Length scale of the simulation domain
c = 1.0                    # Constant for frequency relation
k_min = 2 * np.pi / L      # Minimum wavenumber (based on domain size)
k_max = 10 * k_min         # Maximum wavenumber
amplitude_scale = 0.1      # Scaling factor for mode amplitudes
frame_interval = 50        # Interval between frames in milliseconds

# Generate random wavevectors, amplitudes, and phases
np.random.seed(0)  # Set seed for reproducibility
wavevectors = k_min + (k_max - k_min) * np.random.rand(N)
angles = 2 * np.pi * np.random.rand(N)
kx = wavevectors * np.cos(angles)  # x-component of wavevector
ky = wavevectors * np.sin(angles)  # y-component of wavevector
amplitudes = amplitude_scale * wavevectors**(-5/6)  # Kolmogorov scaling

# Grid setup for visualization
x = np.linspace(0, L, grid_size)
y = np.linspace(0, L, grid_size)
X, Y = np.meshgrid(x, y)

# Generate fixed random phases for each mode
phases = 2 * np.pi * np.random.rand(N)

# Time-dependent velocity field calculation function with fixed phases
def calculate_velocity_field(X, Y, t):
    u = np.zeros(X.shape)  # Initialize u component of velocity
    v = np.zeros(Y.shape)  # Initialize v component of velocity
    for i in range(N):
        # Calculate omega for each mode
        omega = c * np.sqrt(kx[i]**2 + ky[i]**2)
        
        # Use fixed phase for continuity
        u += amplitudes[i] * np.cos(kx[i] * X + ky[i] * Y - omega * t + phases[i])
        v += amplitudes[i] * np.sin(kx[i] * X + ky[i] * Y - omega * t + phases[i])
    
    return u, v


# Set up the figure and quiver plot
fig, ax = plt.subplots(figsize=(6, 6))
u, v = calculate_velocity_field(X, Y, 0)  # Initial velocity field
quiver = ax.quiver(X, Y, u, v, scale=1.5, color='blue')
ax.set_title('Isotropic Turbulent Velocity Field')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.axis('equal')
ax.grid()

# Animation update function
def update(frame):
    t = frame * 0.1  # Time progression for each frame (adjust speed by changing 0.1)
    u, v = calculate_velocity_field(X, Y, t)
    quiver.set_UVC(u, v)  # Update the quiver vectors
    return quiver,

# Create the animation
ani = FuncAnimation(fig, update, frames=100, interval=frame_interval, blit=True)

# Display the animation
HTML(ani.to_jshtml())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Parameters
grid_size = 50             # Grid size for the velocity field (20x20 points)
L = 1.0                    # Length scale of the simulation domain
k_min = 2 * np.pi / L      # Minimum wavenumber (based on domain size)
k_max = 10 * k_min         # Maximum wavenumber
frame_interval = 10        # Interval between frames in milliseconds

# Choose parameters for different conditions
condition = 'still'  # Change this to 'still', 'breezy', or 'stormy'

if condition == 'still':
    amplitude_scale = 0.02
    N = 20
    c = 0.5
elif condition == 'breezy':
    amplitude_scale = 0.05
    N = 50
    c = 1.0
elif condition == 'stormy':
    amplitude_scale = 0.1
    N = 100
    c = 1.5

# Generate random wavevectors, amplitudes, and fixed phases
np.random.seed(0)  # Set seed for reproducibility
wavevectors = k_min + (k_max - k_min) * np.random.rand(N)
angles = 2 * np.pi * np.random.rand(N)
kx = wavevectors * np.cos(angles)
ky = wavevectors * np.sin(angles)
amplitudes = amplitude_scale * wavevectors**(-5/6)
phases = 2 * np.pi * np.random.rand(N)

# Grid setup for visualization
x = np.linspace(0, L, grid_size)
y = np.linspace(0, L, grid_size)
X, Y = np.meshgrid(x, y)

# Time-dependent velocity field calculation function with fixed phases
def calculate_velocity_field(X, Y, t):
    u = np.zeros(X.shape)
    v = np.zeros(Y.shape)
    for i in range(N):
        omega = c * np.sqrt(kx[i]**2 + ky[i]**2)
        u += amplitudes[i] * np.cos(kx[i] * X + ky[i] * Y - omega * t + phases[i])
        v += amplitudes[i] * np.sin(kx[i] * X + ky[i] * Y - omega * t + phases[i])
    return u, v

# Set up the figure and quiver plot
fig, ax = plt.subplots(figsize=(6, 6))
u, v = calculate_velocity_field(X, Y, 0)
quiver = ax.quiver(X, Y, u, v, scale=1.5, color='blue')
ax.set_title(f'{condition.capitalize()} Turbulent Velocity Field')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.axis('equal')
ax.grid()

# Animation update function
def update(frame):
    t = frame * 0.1  # Time progression for each frame
    u, v = calculate_velocity_field(X, Y, t)
    quiver.set_UVC(u, v)  # Update the quiver vectors
    return quiver,

# Create the animation
ani = FuncAnimation(fig, update, frames=100, interval=frame_interval, blit=True)
HTML(ani.to_jshtml())


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation

# Parameters for the streamline visualization
N = 50                     # Number of Fourier modes
grid_size = 20             # Grid size for the velocity field (20x20 points)
L = 1.0                    # Length scale of the simulation domain
c = 1.0                    # Constant for frequency relation
k_min = 2 * np.pi / L      # Minimum wavenumber (based on domain size)
k_max = 10 * k_min         # Maximum wavenumber
amplitude_scale = 0.05     # Scaling factor for mode amplitudes
frame_interval = 50        # Interval between frames in milliseconds

# Generate random wavevectors, amplitudes, and fixed phases
np.random.seed(0)
wavevectors = k_min + (k_max - k_min) * np.random.rand(N)
angles = 2 * np.pi * np.random.rand(N)
kx = wavevectors * np.cos(angles)
ky = wavevectors * np.sin(angles)
amplitudes = amplitude_scale * wavevectors**(-5/6)
phases = 2 * np.pi * np.random.rand(N)

# Grid setup for visualization
x = np.linspace(0, L, grid_size)
y = np.linspace(0, L, grid_size)
X, Y = np.meshgrid(x, y)

# Velocity field calculation function with fixed phases
def calculate_velocity_field(X, Y, t):
    u = np.zeros(X.shape)
    v = np.zeros(Y.shape)
    for i in range(N):
        omega = c * np.sqrt(kx[i]**2 + ky[i]**2)
        u += amplitudes[i] * np.cos(kx[i] * X + ky[i] * Y - omega * t + phases[i])
        v += amplitudes[i] * np.sin(kx[i] * X + ky[i] * Y - omega * t + phases[i])
    return u, v

# Set up the figure for initial streamline plot
fig, ax = plt.subplots(figsize=(6, 6))
u, v = calculate_velocity_field(X, Y, 0)
strm = ax.streamplot(X, Y, u, v, color=np.sqrt(u**2 + v**2), cmap='viridis')
ax.set_title('Turbulent Velocity Field (Streamlines)')
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.axis('equal')

# Animation update function
def update(frame):
    ax.clear()  # Clear the axis to reset the streamplot
    t = frame * 0.1
    u, v = calculate_velocity_field(X, Y, t)
    strm = ax.streamplot(X, Y, u, v, color=np.sqrt(u**2 + v**2), cmap='viridis')
    ax.set_title('Turbulent Velocity Field (Streamlines)')
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.axis('equal')
    return strm

# Create the animation
ani = FuncAnimation(fig, update, frames=100, interval=frame_interval, blit=False)
HTML(ani.to_jshtml())
# Display the animation
# plt.show()
